[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Relationships


## What you will be able to do

Give mapped classes relationships, so that a student holds their enrollments and an enrollment holds
its section, and navigate from one to the other without writing a join. Keep both sides of a
relationship in step with `back_populates`, create related rows from objects alone and let the flush
fill in the foreign keys, and query through a relationship. Recognize the errors from a relationship
with no foreign key, one with two, one side without the other, and a class name the mapper cannot
find.


## The idea

### The problem

**The Session** notebook opened a section for a new course, and had to flush the course first to
learn its id, because a section records its course as a number. Every query that goes from a student
to their courses has joined four tables by hand, and code that wants Chloe Martin's courses passes
the student's id around and asks the database for each course by its id. The classes know which
columns hold foreign keys, and nothing uses that knowledge: `Enrollment` has a `section_id`, and no
way to hand over the section itself.

What the registrar's code wants to write is `chloe.enrollments`, a list, and
`enrollment.section.course .title`, a chain of objects. It wants to make a section for a course that
is not saved yet, give it enrollments for students who are not saved yet, and save the lot in one
step, with every foreign key filled in by whatever put the rows in the database.

### What a relationship is

> A **relationship** is an attribute of a mapped class that holds related objects instead of their
> ids, declared with **`relationship()`** and built from the foreign key between the two tables.
> Seen from the table with the foreign key, it is **many to one**: `Section.course` holds one
> `Course`, and is annotated `Mapped["Course"]`. Seen from the other table, it is **one to many**:
> `Course.sections` holds a list of sections, `Mapped[list["Section"]]`. **`back_populates`** names
> the attribute on the other side, so that a change to one side shows on the other at once. Related
> objects are loaded the first time the attribute is read, which is **lazy loading**, and a flush
> fills in the foreign key columns from the objects the relationships hold.

### Why it works that way

- **The foreign key decides the join.** `relationship()` finds the `ForeignKey` between the two
  tables and joins on it, so a pair of tables with no foreign key, or with two, cannot be joined
  without help, which is two of the Common errors.
- **The annotation decides one or many.** `Mapped[list["Section"]]` is a collection, and
  `Mapped["Course"]` a single object. The quotation marks let a class name a class declared after it.
- **`back_populates` keeps both sides in step in Python.** Setting `section.course` adds the section
  to `course.sections` at once, before any flush, and appending to `course.sections` sets
  `section.course`. Without it, the two sides are two independent attributes that happen to share a
  column.
- **Objects, not ids.** Assigning a course to `section.course` records the object, and the flush
  writes the course's id into `course_id` once the course has one, inserting the course first because
  the foreign key requires it.
- **Adding one object adds what it holds.** A relationship's save-update cascade, on unless told
  otherwise, adds the related objects to the session too, so adding an enrollment adds its student
  and its section, and the section's course.
- **Related objects load when first read.** Reading `chloe.enrollments` runs a `SELECT` the first
  time, and the **Loading Strategies** notebook counts what that costs across a loop.

### Where this shows up

Every ORM application navigates its data this way, and SQLModel, the subject of the
**SQLModel, Deep Dive** guide, declares the same relationships with a `Relationship()` of its own.
The **Many to Many** notebook uses the enrollment between a student and a section,
**Loading Strategies** decides when related objects are loaded, and **Cascades and Deletes** decides
what happens to them when their parent is deleted. The **Joins** notebook of the
**sqlite3, Deep Dive** guide wrote the joins that a relationship writes for you.

### What this notebook covers

- A relationship on each side of every foreign key
- From a student to their courses, without a join
- `back_populates`: both sides in step before any flush
- Objects, not ids: the flush that fills in the keys
- Querying through a relationship
- Which kind of relationship to declare, and how
- A summer term planned from objects alone, finished
- Six errors, from a relationship with no foreign key to a class name misspelled

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import ForeignKey, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship


class Base(DeclarativeBase):
    pass


class Course(Base):
    __tablename__ = "courses"
    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str]
    sections: Mapped[list["Section"]] = relationship(back_populates="course")


class Section(Base):
    __tablename__ = "sections"
    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    course: Mapped["Course"] = relationship(back_populates="sections")


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    drawing = Course(code="ART-100")
    session.add(Section(course=drawing))
    session.commit()
    print(len(drawing.sections), "section | its course_id:", drawing.sections[0].course_id)
```

```
1 section | its course_id: 1
```

The section was given the course as an object, not an id, and the commit saved both, the course
first, with the section's `course_id` filled in from the course's new id. `drawing.sections` held
the section, because `back_populates` links the two sides.


## Setup

Ten imports, and the college built from its `MetaData`.

- `sqlalchemy` is the library itself, and the cell prints its version
- `relationship`, from `sqlalchemy.orm`, declares the relationships, with `DeclarativeBase`,
  `Mapped`, `mapped_column`, `Session` and `sessionmaker`
- `inspect`, from `sqlalchemy`, lists a class's relationships, with `select`, `func`, `insert`,
  `create_engine` and `event`, and what describes a table, `MetaData`, `Table`, `Column`, the types,
  `ForeignKey` and the constraints
- `warnings` catches the warning that one of the Common errors produces
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds the college from `college`, the tables of the **Tables and Metadata** notebook, as the
**SQL Expressions** notebook did, so that the classes can be written below with their relationships.
From the **Many to Many** notebook on, Setup builds the college from these classes.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table, UniqueConstraint,
                        create_engine, event, func, insert, inspect, select)
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, sessionmaker
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### A relationship on each side of every foreign key

The college's classes again, with a relationship on each side of each of the four foreign keys, every
pair linked by `back_populates`. Every collection has an `order_by`, so that a list of enrollments
comes back in the same order every time. `inspect(cls).relationships` lists what each class has:


In [2]:
GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


SessionLocal = sessionmaker(engine)

for cls in (Student, Course, Term, Section, Enrollment):
    for rel in inspect(cls).relationships:
        joined_on = [column.name for column in rel.local_columns]
        print(f"{cls.__name__ + '.' + rel.key:<21} {rel.direction.name:<10} {rel.mapper.class_.__name__:<11} {joined_on}")


Student.enrollments   ONETOMANY  Enrollment  ['id']
Course.sections       ONETOMANY  Section     ['id']
Term.sections         ONETOMANY  Section     ['id']
Section.course        MANYTOONE  Course      ['course_id']
Section.term          MANYTOONE  Term        ['term_id']
Section.enrollments   ONETOMANY  Enrollment  ['id']
Enrollment.student    MANYTOONE  Student     ['student_id']
Enrollment.section    MANYTOONE  Section     ['section_id']


Eight relationships, two for each foreign key. The many-to-one side of each, such as
`Section.course`, joins on the table's own foreign key column, `course_id`, and the one-to-many side,
such as `Course.sections`, on its primary key, `id`, which the other table's foreign key refers to.
`relationship()` read all of that from the `ForeignKey`s the columns already declared.

### From a student to their courses, without a join

Chloe Martin's courses, read by walking from the student to the enrollments, from each enrollment to
its section, and from the section to its term and its course. `echo` is on for the first step:


In [3]:
engine.echo = True
with SessionLocal() as session:
    chloe = session.get(Student, 3)
    print("--- reading chloe.enrollments")
    enrollments = chloe.enrollments
    engine.echo = False
    for enrollment in enrollments:
        section = enrollment.section
        print(f"{section.term.name:<12} {section.course.code:<8} {section.course.title:<27} {enrollment.grade}")


    BEGIN (implicit)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on
    FROM students
    WHERE students.id = ?
    values: (3,)
--- reading chloe.enrollments
    SELECT enrollments.student_id AS enrollments_student_id, enrollments.section_id AS enrollments_section_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE ? = enrollments.student_id ORDER BY enrollments.section_id
    values: (3,)
Fall 2025    CHE-110  General Chemistry           C+
Fall 2025    CSC-201  Data Structures             F
Fall 2025    PSY-101  Introduction to Psychology  B+
Spring 2026  MAT-120  Calculus I                  None
Spring 2026  ENG-105  Composition                 None
Spring 2026  STA-200  Statistics                  None


Reading `chloe.enrollments` sent a `SELECT` for Chloe's enrollments, ordered by section as the
relationship asked, and the loop reached every section, term and course through attributes. None of
the code names a column, and none of it joins, but every `section`, `term` and `course` read for the
first time loaded one more row: the **Loading Strategies** notebook counts those statements, and
shows how to load them all at once.

### back_populates: both sides in step before any flush

A new section is given its course by one side of a relationship, and its term by the other, and both
sides are read back before anything is sent to the database:


In [4]:
with SessionLocal() as session:
    drawing = Course(code="ART-100", title="Drawing", department="Art", credits=3)
    spring = session.scalars(select(Term).where(Term.name == "Spring 2026")).one()

    section = Section(capacity=20)
    section.course = drawing                              # the many-to-one side
    print("the section is in drawing.sections:", section in drawing.sections)

    spring.sections.append(section)                       # the one-to-many side
    print("section.term:", section.term)
    print("course_id and term_id, before a flush:", section.course_id, section.term_id)
    print("in the session:", section in session, drawing in session)
    session.rollback()


the section is in drawing.sections: True
section.term: Term('Spring 2026')
course_id and term_id, before a flush: None None
in the session: True True


Setting `section.course` put the section in `drawing.sections`, and appending it to
`spring.sections` set `section.term`, both at once, in Python. The foreign key columns are still
`None`: nothing has been flushed, and `course_id` and `term_id` are filled in only when the rows are
written. Appending to a term already in the session also brought the section into the session, and
the section brought its course, which is the save-update cascade. The rollback threw the lot away.

### Objects, not ids: the flush that fills in the keys

A new student, a new course and a section of it, and an enrollment that ties them together, made from
objects alone. Only the enrollment is added to the session:


In [5]:
engine.echo = True
with SessionLocal.begin() as session:
    spring = session.scalars(select(Term).where(Term.name == "Spring 2026")).one()
    zoe = Student(name="Zoe Nakamura", email="znakamura@college.edu", program="Computer Science",
                  started_on=date(2026, 1, 12))
    drawing = Course(code="ART-100", title="Drawing", department="Art", credits=3)
    session.add(Enrollment(student=zoe, section=Section(course=drawing, term=spring, capacity=20)))
engine.echo = False


    BEGIN (implicit)
    SELECT terms.id, terms.name, terms.starts_on
    FROM terms
    WHERE terms.name = ?
    values: ('Spring 2026',)
    INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)
    values: ('ART-100', 'Drawing', 'Art', 3)
    INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)
    values: ('Zoe Nakamura', 'znakamura@college.edu', 'Computer Science', '2026-01-12')
    INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)
    values: (11, 4, 20)
    INSERT INTO enrollments (student_id, section_id, grade) VALUES (?, ?, ?) RETURNING status
    values: (26, 41, None)
    COMMIT


One `add()`, and four rows: the enrollment brought its student and its section into the session, and
the section brought its course. The flush put them in the order the foreign keys need, courses and
students first, then the section, whose `course_id` came from the course's new id, 11, then the
enrollment, with Zoe Nakamura's new id, 26, and the section's, 41. **The Session** notebook needed a
flush in the middle to learn a course's id, and here nothing did.

### Querying through a relationship

A relationship attribute also names a join in a query. `.join(Student.enrollments)` joins on the
foreign key the relationship already knows:


In [6]:
MATH_THIS_TERM = (
    select(Course.code, Student.name)
    .join(Student.enrollments)
    .join(Enrollment.section)
    .join(Section.course)
    .where(Section.term_id == 4, Course.department == "Mathematics")
    .order_by(Course.code, Student.name)
)
print(" ".join(str(MATH_THIS_TERM.compile(engine)).split()))

with SessionLocal() as session:
    rows = session.execute(MATH_THIS_TERM).all()
print(len(rows), "enrollments, the first four:", rows[:4])


SELECT courses.code, students.name FROM students JOIN enrollments ON students.id = enrollments.student_id JOIN sections ON sections.id = enrollments.section_id JOIN courses ON courses.id = sections.course_id WHERE sections.term_id = ? AND courses.department = ? ORDER BY courses.code, students.name
21 enrollments, the first four: [('MAT-120', 'Chloe Martin'), ('MAT-120', 'Felix Wagner'), ('MAT-120', 'Isabel Costa'), ('MAT-120', 'Maya Patel')]


The `ON` clauses came from the relationships, in the direction each one points. This is the way to
query across tables with the ORM, and the **Joins and Aggregates** notebook builds on it with counts
and averages.

### Which kind of relationship to declare, and how

| Use | When | Why |
|---|---|---|
| `Mapped["Course"]` on the side with the foreign key | each row belongs to one parent, such as a section to its course | many to one, a single object |
| `Mapped[list["Section"]]` on the other side | a parent has any number of children | one to many, a list |
| `back_populates` on both sides | always, for a pair | both sides change together, and nothing is declared twice |
| `backref="course"` on one side only | older code | creates the other side for you, and cannot be mixed with `back_populates` on the same pair |
| `order_by=` on a collection | any collection that is printed or shown in order | a list whose order the database could otherwise choose |
| `foreign_keys=` | two foreign keys between the same two tables | says which one this relationship follows |

The default is a pair, one on each side, with `back_populates` naming each other and an `order_by` on
the collection.

### A summer term planned from objects alone, finished

The pieces of this notebook in one function. `plan_summer` looks up every course and student it needs
first, then makes a Summer 2026 term, a section of every course it is given, and an enrollment for
every student listed for that course, connecting them all through relationships with no query in
between, and never touching an id. The report comes back through the same relationships:


In [7]:
def plan_summer(SessionLocal, offerings):
    """Open a Summer 2026 term with a section of every course in `offerings`, enrolling the students listed
    for each by email, all through relationships; return what the term holds, read back through them."""
    with SessionLocal.begin() as session:
        courses = {course.code: course for course in session.scalars(select(Course).where(Course.code.in_(offerings)))}
        students = {student.email: student for student in session.scalars(select(Student))}

        summer = Term(name="Summer 2026", starts_on=date(2026, 6, 1))
        session.add(summer)                                   # first, so whatever joins it joins the session
        for code, emails in offerings.items():
            section = Section(course=courses[code], capacity=15)
            summer.sections.append(section)
            for email in sorted(emails, key=lambda email: students[email].name):
                section.enrollments.append(Enrollment(student=students[email]))

        session.flush()
        return {section.course.code: (section.id, [enrollment.student.name for enrollment in section.enrollments])
                for section in summer.sections}



SUMMER = {
    "STA-200": ["areyes@college.edu", "bokafor@college.edu", "znakamura@college.edu"],
    "ART-100": ["cmartin@college.edu", "znakamura@college.edu"],
}
for code, (section_id, names) in plan_summer(SessionLocal, SUMMER).items():
    print(f"{code}  section {section_id}:", names)

with SessionLocal() as session:
    zoe = session.scalars(select(Student).where(Student.email == "znakamura@college.edu")).one()
    print("Zoe Nakamura's sections:", [(enrollment.section.term.name, enrollment.section.course.code)
                                        for enrollment in zoe.enrollments])


STA-200  section 42: ['Ana Reyes', 'Ben Okafor', 'Zoe Nakamura']
ART-100  section 43: ['Chloe Martin', 'Zoe Nakamura']
Zoe Nakamura's sections: [('Spring 2026', 'ART-100'), ('Summer 2026', 'STA-200'), ('Summer 2026', 'ART-100')]


The term held both sections, and every section its students, before a single id existed: the flush
in the function gave them all ids, in an order the foreign keys allow, and the report read them back
through the relationships. Zoe Nakamura's enrollments, read from the other side, now include both
summer sections beside the Spring 2026 drawing class. The lookups come first so that no query runs
while the term is half built, since a query would autoflush it, and the term is added to the session
first, so every section appended to it, and every enrollment appended to a section, joins the session
at once. The fourth of the Common errors shows what either mistake costs.

### Where each part came from

| In `plan_summer` | What it relies on | The section that showed it |
|---|---|---|
| `summer.sections.append(section)` | the one-to-many side, which sets `section.term` | back_populates: both sides in step before any flush |
| `Section(course=course, ...)` and `Enrollment(student=student)` | the many-to-one side, given an object | Objects, not ids: the flush that fills in the keys |
| `session.add(summer)`, before anything is appended | the save-update cascade, adding everything the term holds | Objects, not ids: the flush that fills in the keys |
| `session.flush()` before the report | ids given to every new row, in foreign key order | Objects, not ids: the flush that fills in the keys |
| `section.course.code`, `enrollment.student.name` | navigation through relationships | From a student to their courses, without a join |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/11-relationships-solutions.ipynb).

**1.** Through `course.sections`, print every section of Calculus I, `MAT-120`, with its term's name.


In [8]:
# your code here


**2.** Through `section.enrollments`, list the names of the students in section 36, Data Structures
in Spring 2026.


In [9]:
# your code here


**3.** Give Grace Lin, student 7, an enrollment in Spring 2026's Statistics, section 40, by appending
to `grace.enrollments`, and show that the section's list of enrollments holds it before any flush.
Roll back at the end.


In [10]:
# your code here


**4.** With `.join(Student.enrollments)`, count the Spring 2026 enrollments of every program.


In [11]:
# your code here


**5.** Print, for every relationship of `Enrollment`, its direction and the class at its other end.


In [12]:
# your code here


**6.** Show, through `term.sections`, how many students are enrolled in each section of Summer 2026,
and how many in the whole term.


In [13]:
# your code here


## Common errors

### sqlalchemy.exc.NoForeignKeysError: Could not determine join condition between parent/child tables on relationship Room.bookings - there are no foreign keys linking these tables.  Ensure that referencing columns are associated with a ForeignKey or ForeignKeyConstraint, or specify a 'primaryjoin' expression.


In [14]:
class RoomBase(DeclarativeBase):
    pass


class Room(RoomBase):
    __tablename__ = "rooms"

    id: Mapped[int] = mapped_column(primary_key=True)
    bookings: Mapped[list["Booking"]] = relationship(back_populates="room")


class Booking(RoomBase):
    __tablename__ = "bookings"

    id: Mapped[int] = mapped_column(primary_key=True)
    room_id: Mapped[int]                                   # a room's id, but no ForeignKey
    room: Mapped["Room"] = relationship(back_populates="bookings")


Room()


NoForeignKeysError: Could not determine join condition between parent/child tables on relationship Room.bookings - there are no foreign keys linking these tables.  Ensure that referencing columns are associated with a ForeignKey or ForeignKeyConstraint, or specify a 'primaryjoin' expression.

`room_id` holds a room's id, and nothing says so: without a `ForeignKey`, `relationship()` has no way
to know which column joins the two tables. The error came when the class was first used, `Room()`,
because relationships are set up then, all at once. Declare the foreign key:


In [15]:
class RoomBase(DeclarativeBase):
    pass


class Room(RoomBase):
    __tablename__ = "rooms"

    id: Mapped[int] = mapped_column(primary_key=True)
    bookings: Mapped[list["Booking"]] = relationship(back_populates="room")


class Booking(RoomBase):
    __tablename__ = "bookings"

    id: Mapped[int] = mapped_column(primary_key=True)
    room_id: Mapped[int] = mapped_column(ForeignKey("rooms.id"))
    room: Mapped["Room"] = relationship(back_populates="bookings")


room = Room()
room.bookings.append(Booking())
print(room.bookings[0].room is room)


True


### sqlalchemy.exc.AmbiguousForeignKeysError: Could not determine join condition between parent/child tables on relationship Lecture.instructor - there are multiple foreign key paths linking the tables.  Specify the 'foreign_keys' argument, providing a list of those columns which should be counted as containing a foreign key reference to the parent table.


In [16]:
class StaffBase(DeclarativeBase):
    pass


class Person(StaffBase):
    __tablename__ = "people"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


class Lecture(StaffBase):
    __tablename__ = "lectures"

    id: Mapped[int] = mapped_column(primary_key=True)
    instructor_id: Mapped[int] = mapped_column(ForeignKey("people.id"))
    substitute_id: Mapped[int | None] = mapped_column(ForeignKey("people.id"))
    instructor: Mapped["Person"] = relationship()


Lecture()


AmbiguousForeignKeysError: Could not determine join condition between parent/child tables on relationship Lecture.instructor - there are multiple foreign key paths linking the tables.  Specify the 'foreign_keys' argument, providing a list of those columns which should be counted as containing a foreign key reference to the parent table.

A lecture refers to people twice, once for its instructor and once for a substitute, so there are two
foreign keys between the tables and `relationship()` refused to pick one. `foreign_keys` says which
column each relationship follows:


In [17]:
class StaffBase(DeclarativeBase):
    pass


class Person(StaffBase):
    __tablename__ = "people"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


class Lecture(StaffBase):
    __tablename__ = "lectures"

    id: Mapped[int] = mapped_column(primary_key=True)
    instructor_id: Mapped[int] = mapped_column(ForeignKey("people.id"))
    substitute_id: Mapped[int | None] = mapped_column(ForeignKey("people.id"))
    instructor: Mapped["Person"] = relationship(foreign_keys=[instructor_id])
    substitute: Mapped["Person | None"] = relationship(foreign_keys=[substitute_id])


lecture = Lecture(instructor=Person(name="Dr. Okafor"), substitute=Person(name="Ms. Lin"))
print(lecture.instructor.name, "|", lecture.substitute.name)


Dr. Okafor | Ms. Lin


### No error, and one side out of step: two relationships without back_populates


In [18]:
class PairBase(DeclarativeBase):
    pass


class Club(PairBase):
    __tablename__ = "clubs"

    id: Mapped[int] = mapped_column(primary_key=True)
    members: Mapped[list["Member"]] = relationship()               # no back_populates


class Member(PairBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    club_id: Mapped[int] = mapped_column(ForeignKey("clubs.id"))
    club: Mapped["Club"] = relationship()                           # and none here


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    chess = Club()
    ana = Member(club=chess)
    print("ana is in chess.members:", ana in chess.members)
for warning in caught:
    print(type(warning.message).__name__ + ":", str(warning.message).split(" If this is")[0])


ana is in chess.members: False


The two relationships use the same foreign key, and nothing tells SQLAlchemy they are two sides of
one link, so setting `ana.club` left `chess.members` empty, and a program reading the club's members
before a flush and a reload would not find Ana. SQLAlchemy noticed when it set the classes up, and
warned that both relationships write the same column, which the cell caught and printed in part,
since the full message runs on to name `back_populates` as the answer:


In [19]:
class PairBase(DeclarativeBase):
    pass


class Club(PairBase):
    __tablename__ = "clubs"

    id: Mapped[int] = mapped_column(primary_key=True)
    members: Mapped[list["Member"]] = relationship(back_populates="club")


class Member(PairBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    club_id: Mapped[int] = mapped_column(ForeignKey("clubs.id"))
    club: Mapped["Club"] = relationship(back_populates="members")


chess = Club()
ana = Member(club=chess)
print("ana is in chess.members:", ana in chess.members)


ana is in chess.members: True


### sqlalchemy.exc.SAWarning: Object of type <Section> not in session, add operation along 'Course.sections' will not proceed


In [20]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    with SessionLocal() as session:
        statistics = session.scalars(select(Course).where(Course.code == "STA-200")).one()
        fall = Term(name="Fall 2026", starts_on=date(2026, 8, 24))
        section = Section(course=statistics, term=fall, capacity=25)     # given a course that is in the session
        session.scalars(select(Student).limit(1)).all()                  # any query autoflushes
        print("the section is in the session:", section in session)
        session.rollback()
for warning in caught:
    print(type(warning.message).__name__ + ":", str(warning.message).split(" (This warning")[0])


the section is in the session: False


The new section was given a course that the session holds, and `back_populates` put it in
`statistics.sections`. That does not add it to the session: SQLAlchemy 2.0 brings a new object into
a session only through the object it was added to, appended to, or assigned to, and never from the
other end of a `back_populates`. The next query autoflushed, found a section it could not insert
along `Course.sections`, and warned instead. The cell caught the warning, since a warning printed
the ordinary way carries the name of a temporary file. Add the new object itself, which brings its
term with it, or append it to something the session holds, as `plan_summer` appends every section to
its term:


In [21]:
with SessionLocal() as session:
    statistics = session.scalars(select(Course).where(Course.code == "STA-200")).one()
    fall = Term(name="Fall 2026", starts_on=date(2026, 8, 24))
    section = Section(course=statistics, term=fall, capacity=25)
    session.add(section)                                              # and with it, the term it was given
    session.scalars(select(Student).limit(1)).all()
    print("the section is in the session:", section in session, "| its id after the autoflush:", section.id)
    session.rollback()


the section is in the session: True | its id after the autoflush: 44


### sqlalchemy.exc.ArgumentError: Error creating backref 'club' on relationship 'Club.members': property of that name exists on mapper 'Mapper[Member(members)]'


In [22]:
class MixedBase(DeclarativeBase):
    pass


class Club(MixedBase):
    __tablename__ = "clubs"

    id: Mapped[int] = mapped_column(primary_key=True)
    members: Mapped[list["Member"]] = relationship(backref="club")         # creates Member.club


class Member(MixedBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    club_id: Mapped[int] = mapped_column(ForeignKey("clubs.id"))
    club: Mapped["Club"] = relationship(back_populates="members")          # and so does this


Member()


ArgumentError: Error creating backref 'club' on relationship 'Club.members': property of that name exists on mapper 'Mapper[Member(members)]'

`backref` is the older way of declaring a pair: one relationship with `backref="club"` creates the
other side itself. `Member` had already declared `club`, so the backref found the name taken. Use one
style for a pair: `back_populates` on both sides, as every class in this guide does, which keeps both
attributes visible in the classes that own them.

### sqlalchemy.exc.InvalidRequestError: When initializing mapper Mapper[Section(sections)], expression 'Cours' failed to locate a name ('Cours'). If this is a class name, consider adding this relationship() to the <class '__main__.Section'> class after both dependent classes have been defined.


In [23]:
class TypoBase(DeclarativeBase):
    pass


class Course(TypoBase):
    __tablename__ = "courses"

    id: Mapped[int] = mapped_column(primary_key=True)


class Section(TypoBase):
    __tablename__ = "sections"

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    course: Mapped["Cours"] = relationship()


Section()


InvalidRequestError: When initializing mapper Mapper[Section(sections)], expression 'Cours' failed to locate a name ('Cours'). If this is a class name, consider adding this relationship() to the <class '__main__.Section'> class after both dependent classes have been defined.

The name in quotation marks is looked up when the classes are set up, among the classes of the same
base, and there is no `Cours`. The quotation marks are what let `Section` name a class declared
after it, and they also mean no editor or Python itself checks the name as it is typed. The message
suggests declaring the relationship later, which is for a class that really is missing; here the fix
is the spelling. The cell also rebound `Course` and `Section` to these practice classes, and nothing
after it uses the college's.

Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [24]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `relationship()` gives a class an attribute of related objects, built from the foreign key: many
  to one, `Mapped["Course"]`, on the side with the key, and one to many, `Mapped[list["Section"]]`,
  on the other.
- `back_populates` on both sides keeps them in step in Python, before any flush.
- Assigning objects instead of ids is enough: the flush fills in the foreign keys, in an order the
  keys allow, and adding one object adds what its relationships hold.
- A relationship is also a join in a query, `.join(Student.enrollments)`, and its objects load when
  first read.
- No foreign key, two foreign keys, a missing `back_populates`, a new object that never joined the
  session, and a misspelled class name are the mistakes that relationships make easy.


## What is next

The **Many to Many** notebook links students to sections through their enrollments: `secondary`, the
association table, and the extra column it cannot reach.


---

&#8592; **Previous:** [The Identity Map](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/10-the-identity-map.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Many to Many](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/12-many-to-many.ipynb) &#8594;
